# 最高精度ベースラインから改善 6 パターン提出

**ベースライン（現最高）**: 4本ブレンド 0.20 : 0.20 : **0.45** : 0.15（count1, BPR128, **stacking**, 4本目）

**流れ**:
1. **新しい特徴を 1 つ追加**し、それで **モデルを 1 本作成**（現土台＝BPR64+count1 に TF-IDF SVD を足して LGB 学習）。
2. その予測を **既存 4 本とスタッキング**（ブレンド）。**全ファイルでスタッキング比率は 0.45 に統一**。
3. 「新モデル」の重みだけ 6 パターン変えて **6 個の提出ファイル**を出力。

| # | 新モデルの重み | 提出ファイル |
|---|---------------|-------------|
| 1 | 5%  | submission_baseline_improve_01.csv |
| 2 | 8%  | submission_baseline_improve_02.csv |
| 3 | 10% | submission_baseline_improve_03.csv |
| 4 | 12% | submission_baseline_improve_04.csv |
| 5 | 15% | submission_baseline_improve_05.csv |
| 6 | 18% | submission_baseline_improve_06.csv |

## 1. セットアップ（既存 4 本のパス）

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from lib.improvement_candidates import get_setup
from lib.submission import verify_submission, blend_n_submissions, blend_two_submissions

ctx = get_setup(seed=42, output_dir="outputs", use_best_pipeline=True)
test_ids = ctx.test["ID"].values

path_count1 = ctx.submissions_dir / "submission_2hop_bpr64_count1.csv"
path_bpr128 = ctx.submissions_dir / "submission_2hop_bpr128_only.csv"
path_stacking = ctx.submissions_dir / "submission_improvement_03_stacking.csv"

path_4th = None
for cand in ["submission_similar_movies_reviewed.csv", "submission_improvement_05_scale_pos_weight.csv", "submission_blend_bpr64_count1_bpr128.csv"]:
    p = ctx.submissions_dir / cand
    if p.exists():
        path_4th = p
        print(f"  4本目: {cand}")
        break
if path_4th is None:
    path_2way = ctx.submissions_dir / "submission_blend_bpr64_count1_bpr128.csv"
    blend_two_submissions(path_count1, path_bpr128, path_2way, weight_a=0.65, test_ids=test_ids)
    path_4th = path_2way
    print(f"  4本目用に作成: {path_4th.name}")

for name, p in [("count1", path_count1), ("BPR128", path_bpr128), ("stacking", path_stacking), ("4本目", path_4th)]:
    print(f"  [{name}] {'OK' if p.exists() else 'なし'}: {p.name}")
print(f"提出先: {ctx.submissions_dir}")

  4本目: submission_blend_bpr64_count1_bpr128.csv
  [count1] OK: submission_2hop_bpr64_count1.csv
  [BPR128] OK: submission_2hop_bpr128_only.csv
  [stacking] OK: submission_improvement_03_stacking.csv
  [4本目] OK: submission_blend_bpr64_count1_bpr128.csv
提出先: outputs/submissions


## 2. 新しい特徴を追加してモデルを 1 本作成

現土台（BPR64 + count1）に **movie_info の TF-IDF → SVD** を 1 塊追加し、LGB で学習して提出 CSV を 1 本出力する。

In [3]:
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

from lib.improvement_candidates import get_bpr_base
from lib.two_hop import add_2hop_features, TWO_HOP_REVIEW_COUNT
from lib.submission import save_submission

train_c1, test_c1, feats_c1 = get_bpr_base(ctx, factors=64)
add_2hop_features(train_c1, test_c1, columns=[TWO_HOP_REVIEW_COUNT])
feats_c1 = feats_c1 + [TWO_HOP_REVIEW_COUNT]

n_tfidf, n_svd = 200, 12
text_tr = train_c1["movie_info"].astype(str).replace("nan", "")
text_te = test_c1["movie_info"].astype(str).replace("nan", "")
tfidf = TfidfVectorizer(max_features=n_tfidf, min_df=2, max_df=0.95, ngram_range=(1, 2), sublinear_tf=True)
svd = TruncatedSVD(n_components=n_svd, random_state=42)
M_tr = svd.fit_transform(tfidf.fit_transform(text_tr))
M_te = svd.transform(tfidf.transform(text_te))

tfidf_cols = [f"tfidf_svd_{i}" for i in range(n_svd)]
for i, c in enumerate(tfidf_cols):
    train_c1[c] = M_tr[:, i]
    test_c1[c] = M_te[:, i]

feats_new = feats_c1 + tfidf_cols
X_tr = train_c1[feats_new].copy()
X_te = test_c1[feats_new].copy()
for col in feats_new:
    if not pd.api.types.is_numeric_dtype(X_tr[col]) and not pd.api.types.is_categorical_dtype(X_tr[col]):
        X_tr[col] = X_tr[col].astype("category")
        X_te[col] = X_te[col].astype("category")

model = lgb.LGBMClassifier(**ctx.lgb_params)
model.fit(X_tr, ctx.y)
pred_new = model.predict_proba(X_te)[:, 1]

path_new = ctx.submissions_dir / "submission_baseline_new_tfidf.csv"
save_submission(ctx.test, pred_new, path_new, sanitize=True)
print(f"  新モデル（BPR64+count1+TF-IDF SVD）→ {path_new.name}")

  0%|          | 0/100 [00:00<?, ?it/s]

  新モデル（BPR64+count1+TF-IDF SVD）→ submission_baseline_new_tfidf.csv


## 3. 5 本でスタッキング（stacking=0.45 固定・新モデル重み 6 パターン）

count1, BPR128, **stacking**, 4本目, **新モデル** の 5 本をブレンド。**スタッキングは常に 0.45**。残り 0.55 のうち新モデル分を除いたものを count1:BPR128:4本目 = 0.20:0.20:0.15 の比で配分。

In [9]:
path_new = ctx.submissions_dir / "submission_baseline_new_tfidf.csv"
if not path_new.exists():
    raise FileNotFoundError("先にセル「2. 新しい特徴を追加してモデルを 1 本作成」を実行してください")

paths_5 = [path_count1, path_bpr128, path_stacking, path_4th, path_new]
# 空ファイルがないか事前チェック（EmptyDataError 対策）
for p in paths_5:
    if p.stat().st_size == 0:
        raise FileNotFoundError(f"空のファイルです。再生成するか削除してください: {p.name}")

STACKING_W = 0.45  # 全ファイルで同じスタッキング比率
# count1 : BPR128 : 4本目 の比をベースラインに合わせる = 0.20 : 0.20 : 0.15 → 4:4:3
RATIO_4_4_3 = [4 / 11, 4 / 11, 3 / 11]  # count1, bpr128, 4th

patterns = []
for new_w in [0.05, 0.08, 0.10, 0.12, 0.15, 0.18]:
    rest = 0.55 - new_w  # count1 + bpr128 + 4th
    w_count1 = rest * RATIO_4_4_3[0]
    w_bpr128 = rest * RATIO_4_4_3[1]
    w_4th = rest * RATIO_4_4_3[2]
    weights = [w_count1, w_bpr128, STACKING_W, w_4th, new_w]
    patterns.append((weights, f"submission_baseline_improve_{len(patterns)+1:02d}.csv"))

for weights, out_name in patterns:
    out_path = ctx.submissions_dir / out_name
    r = blend_n_submissions(paths_5, weights, out_path, test_ids=test_ids)
    status = "OK" if r.get("ok") else r.get("message", "")
    new_pct = weights[4] * 100
    print(f"  新{new_pct:.0f}% → {out_name}  ({status})")

  新5% → submission_baseline_improve_01.csv  (OK)
  新8% → submission_baseline_improve_02.csv  (OK)
  新10% → submission_baseline_improve_03.csv  (OK)
  新12% → submission_baseline_improve_04.csv  (OK)
  新15% → submission_baseline_improve_05.csv  (OK)
  新18% → submission_baseline_improve_06.csv  (OK)


## 4. 提出ファイル一覧

In [10]:
out_names = [f"submission_baseline_improve_{i:02d}.csv" for i in range(1, 7)]
for name in out_names:
    p = ctx.submissions_dir / name
    if p.exists():
        v = verify_submission(p, ctx.test)
        print(f"  {name}  ({v.get('message', '')})")
    else:
        print(f"  {name}  (未生成)")

  submission_baseline_improve_01.csv  (OK)
  submission_baseline_improve_02.csv  (OK)
  submission_baseline_improve_03.csv  (OK)
  submission_baseline_improve_04.csv  (OK)
  submission_baseline_improve_05.csv  (OK)
  submission_baseline_improve_06.csv  (OK)
